In [1]:
import carla
import random
import math

In [2]:
client = carla.Client('localhost', 2000)
world = client.get_world()
print(world)

World(id=3205323787809300769)


In [3]:
print(world.get_map().name)

Carla/Maps/Town10HD_Opt


In [4]:
# remove all the layers of the map except for the road itself.
# (the road layout isn't a toggleable layer, so it stays loaded; layers only work on "_Opt" maps like Town10HD_Opt)
# Unload one layer at a time: combining them with `|` yields an int, which the API rejects.
for layer in [
    carla.MapLayer.Buildings,
    carla.MapLayer.Decals,
    carla.MapLayer.Foliage,
    carla.MapLayer.Ground,
    carla.MapLayer.ParkedVehicles,
    carla.MapLayer.Particles,
    carla.MapLayer.Props,
    carla.MapLayer.StreetLights,
    carla.MapLayer.Walls,
]:
    world.unload_map_layer(layer)

# Traffic lights, traffic signs and the remaining buildings are part of the base level, not toggleable layers,
# so hide them individually as environment objects.
ids_to_hide = set()
for label in [
    carla.CityObjectLabel.TrafficLight,
    carla.CityObjectLabel.TrafficSigns,
    carla.CityObjectLabel.Poles,  # sign / traffic light poles
    carla.CityObjectLabel.Buildings,
]:
    ids_to_hide.update(obj.id for obj in world.get_environment_objects(label))
world.enable_environment_objects(ids_to_hide, False)
print(f"Hid {len(ids_to_hide)} environment objects")

Hid 232 environment objects


In [5]:
# spawn a car at a fixed recommended spawn point (same car + location every run)
VEHICLE_TYPE = 'vehicle.tesla.model3'
SPAWN_INDEX = 5

# re-running would collide with the previous car at the same spot, so remove it first
try:
    vehicle.destroy()
except NameError:
    pass

vehicle_bp = world.get_blueprint_library().find(VEHICLE_TYPE)

spawn_points = world.get_map().get_spawn_points()
spawn_point = spawn_points[SPAWN_INDEX]

vehicle = world.spawn_actor(vehicle_bp, spawn_point)
print(f"Spawned {vehicle.type_id} (id {vehicle.id}) at spawn point {SPAWN_INDEX}: {spawn_point.location}")

# move the spectator camera above the car so it's visible
spectator = world.get_spectator()
spectator.set_transform(carla.Transform(
    spawn_point.location + carla.Location(z=30),
    carla.Rotation(pitch=-90),
))

Spawned vehicle.tesla.model3 (id 24) at spawn point 5: Location(x=-106.649590, y=-17.073978, z=0.600000)


In [6]:
# place forward facing camera attached to the ego vehicle
import queue

# re-running would leave the old camera attached, so remove it first
try:
    camera.stop()
    camera.destroy()
except NameError:
    pass

camera_bp = world.get_blueprint_library().find('sensor.camera.rgb')
camera_bp.set_attribute('image_size_x', '640')
camera_bp.set_attribute('image_size_y', '480')
camera_bp.set_attribute('fov', '90')

# mounted on the hood/windshield area, looking straight ahead
camera_transform = carla.Transform(carla.Location(x=1.5, z=2.4))
camera = world.spawn_actor(camera_bp, camera_transform, attach_to=vehicle)

# the callback runs on another thread, so just hand images to the main loop through a queue
image_queue = queue.Queue()
camera.listen(image_queue.put)
print(f"Spawned camera (id {camera.id}) attached to vehicle {vehicle.id}")

Spawned camera (id 25) attached to vehicle 24


In [ ]:
# visualize waypoints
# for wp in world.get_map().generate_waypoints(2.0):  # one every 2 m on all lanes
#     world.debug.draw_point(wp.transform.location, size=0.1, life_time=120)

In [ ]:
# start autopilot 
import os
import csv
import shutil
import numpy as np
from PIL import Image

carla_map = world.get_map()

def get_speed():
    v = vehicle.get_velocity()
    speed_ms = math.sqrt(v.x**2 + v.y**2 + v.z**2)
    return speed_ms

def get_steering_angle(x: float) -> float:
    """Return VehicleControl.steer in [-1, 1] that points the car at the leftmost lane waypoint x meters ahead."""
    waypoint = carla_map.get_waypoint(vehicle.get_location())
    
    # get waypoints x meters ahead (next() returns a list; at junctions it has several branches)
    candidates = waypoint.next(x)
    if not isinstance(candidates, (list, tuple)):
        candidates = [candidates]
    transform = vehicle.get_transform()

    def heading_error(w):
        # signed angle (deg) from the vehicle's heading to waypoint w.
        # CARLA yaw is clockwise-positive, so positive = target is to the right, negative = to the left
        loc = w.transform.location
        desired_yaw = math.degrees(math.atan2(loc.y - transform.location.y, loc.x - transform.location.x))
        error = desired_yaw - transform.rotation.yaw
        return (error + 180) % 360 - 180  # wrap to [-180, 180]

    # leftmost waypoint = the one with the smallest (most negative) signed angle
    error = min(heading_error(w) for w in candidates)
    print(waypoint.road_id, waypoint.lane_id, waypoint.is_junction, len(candidates), error)  # DEBUG
    # normalize by the front wheels' max steer angle (degrees) -> steer in [-1, 1]
    max_steer = vehicle.get_physics_control().wheels[0].max_steer_angle
    return max(-1.0, min(1.0, error / max_steer))

def save_cropped(image, path):
    """Save a carla.Image to disk with the top CROP_TOP_FRACTION (the sky) removed."""
    bgra = np.frombuffer(image.raw_data, dtype=np.uint8).reshape(image.height, image.width, 4)
    rgb = bgra[int(image.height * CROP_TOP_FRACTION):, :, 2::-1]  # drop the sky rows, BGRA -> RGB
    Image.fromarray(rgb).save(path)

MAX_SPEED = 300
TARGET_SPEED = 10
STABLE_TOLERANCE = 1  # recording starts once |speed - TARGET_SPEED| <= this (same units as get_speed)
LOOKAHEAD = 5.0  # meters ahead of the car to aim at
GAIN = 1.0  # scales how aggressively throttle responds to speed error
DAMP = 0.5  # damping on speed change: raise to reduce oscillation, lower for a faster rise
CROP_TOP_FRACTION = 0.5  # fraction of the image height removed from the top (the sky)

# recording output: images/<id>.png plus steering.csv, linked by the id column
DATA_DIR = 'training_data'
IMAGE_DIR = os.path.join(DATA_DIR, 'images')
CSV_PATH = os.path.join(DATA_DIR, 'steering.csv')

# wipe any previous recording (images folder and csv) before starting a new one
shutil.rmtree(IMAGE_DIR, ignore_errors=True)
if os.path.exists(CSV_PATH):
    os.remove(CSV_PATH)
os.makedirs(IMAGE_DIR, exist_ok=True)

# drop any frames captured before the run starts
while not image_queue.empty():
    image_queue.get_nowait()

recording = False
record_id = 0
steer_by_frame = {}  # sim frame -> steer applied for that frame
pending_images = {}  # sim frame -> image waiting for its steer value
csv_file = open(CSV_PATH, 'w', newline='')
csv_writer = csv.writer(csv_file)
csv_writer.writerow(['id', 'image', 'frame', 'steering'])

throttle = 0
prev_speed = get_speed()

try:
    while True:
        snapshot = world.wait_for_tick()  # only update once per simulator tick

        # id controller for speed
        curr_speed = get_speed()
        throttle += GAIN * (TARGET_SPEED - curr_speed) / MAX_SPEED - DAMP * (curr_speed - prev_speed)
        prev_speed = curr_speed
        throttle = max(0, min(1, throttle))  # cap throttle to [0, 1]
        steer = get_steering_angle(LOOKAHEAD)
        vehicle.apply_control(carla.VehicleControl(throttle=throttle, steer=steer))
        print(curr_speed, steer)

        # start recording as soon as the speed is stable at the target (stays on afterwards)
        if not recording and abs(curr_speed - TARGET_SPEED) <= STABLE_TOLERANCE:
            recording = True
            print(f"Speed stable at {curr_speed:.2f}, recording started")

        # collect camera frames; match each with the steer computed for the same sim frame
        while not image_queue.empty():
            image = image_queue.get_nowait()
            if recording:
                pending_images[image.frame] = image
        if recording:
            steer_by_frame[snapshot.frame] = steer
            for frame in sorted(pending_images):
                if frame in steer_by_frame:
                    image = pending_images.pop(frame)
                    name = f'{record_id:06d}.png'
                    save_cropped(image, os.path.join(IMAGE_DIR, name))
                    csv_writer.writerow([record_id, name, frame, steer_by_frame.pop(frame)])
                    record_id += 1
            csv_file.flush()
            # discard bookkeeping for frames that will never match
            for frame in [f for f in steer_by_frame if f < snapshot.frame - 20]:
                del steer_by_frame[frame]
            for frame in [f for f in pending_images if f < snapshot.frame - 20]:
                del pending_images[frame]
finally:
    csv_file.close()
    print(f"Saved {record_id} frames to {DATA_DIR}")
